# Doodle Alive: free LTX-Video GPU server (Kaggle or Colab)

Runs the same LTX-Video model as the Hugging Face Space on a **free notebook GPU**, behind a public
`https://….gradio.live` link with the **same API** (`/image_to_video`). The Doodle Alive backend uses it
automatically when every Hugging Face key is out of quota.

## Kaggle (recommended: about 30 free GPU hours a week)
1. kaggle.com → **Create → New Notebook → File → Import Notebook** → upload this file.
2. Right panel → **Settings**: *Accelerator* = **GPU T4 x2** (needs both GPUs: one for the text encoder, one for video), *Internet* = **On** (needs a phone-verified account).
3. To keep it running up to 12 h with the tab closed: **Save Version → Save & Run All (Commit)**, then open the
   version's **Logs** and copy the `https://….gradio.live` link. (Or just **Run All** while you keep the tab open.)
4. Paste the link in the Doodle Alive admin panel (page URL + `#admin`) → *Free GPU servers* → **Kaggle** → Save.

## Colab (backup)
File → Upload notebook → *Runtime → Change runtime type* → **T4 GPU** → *Run all* → paste the link under **Colab**.
Keep the tab open; free Colab disconnects when idle.

Each start downloads the models (~15 GB; the cell prints how long each part took). Watch progress in the gradio page or under the last cell; a T4 takes a few minutes per video.

In [ ]:
%pip install -q -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg gradio
# The preinstalled torchao is too old for the latest diffusers (ImportError: FqnToConfig); it isn't used here.
%pip uninstall -q -y torchao

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')  # less memory fragmentation

import random
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import torch
from diffusers import LTXConditionPipeline
from diffusers.utils import export_to_video
from huggingface_hub import snapshot_download
from PIL import Image
from transformers import T5EncoderModel

assert torch.cuda.is_available(), 'No GPU: turn on the GPU accelerator in the notebook settings'

# Re-running this cell (or re-importing the notebook) keeps the old models in GPU memory: free them first.
import gc
for _old in ('pipe', 'text_encoder'):
    globals().pop(_old, None)
gc.collect()
torch.cuda.empty_cache()
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))

# T4 / P100 have no bfloat16 hardware: bf16 is emulated and many times slower there.
# LTX-Video must not run in float16 (it overflows: striped/mosaic textures over the whole video),
# so on those GPUs the video model runs in float32 (~8 GB, still fits a T4). The text encoder stays bf16.
native_bf16 = torch.cuda.get_device_capability(0)[0] >= 8
MODEL_DTYPE = torch.bfloat16 if native_bf16 else torch.float32

# The text encoder (T5-XXL, ~9.5 GB) and the video model (~5 GB) don't fit one 15 GB T4 together.
# Kaggle 'T4 x2': text encoder on GPU 1, video model on GPU 0. One GPU (Colab): they take turns.
MODEL_ID = 'Lightricks/LTX-Video-0.9.5'  # 2B: the 13B 0.9.7/0.9.8 models don't fit a free GPU
VIDEO_DEVICE = 'cuda:0'
TEXT_DEVICE = 'cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0'
SHARED_GPU = TEXT_DEVICE == VIDEO_DEVICE

# LTX's own text_encoder folder is T5-XXL saved in float32 (~19 GB to download, then converted to bf16
# on the slow CPU). The same T5 v1.1 XXL encoder saved in bf16 is half the download and loads as-is.
TEXT_ENCODER_ID = 'city96/t5-v1_1-xxl-encoder-bf16'


def _prefetch(fn, *args, **kwargs):
    try:
        fn(*args, **kwargs)
    except Exception as exc:  # loading below downloads (or falls back) on its own
        print(f'prefetch failed, will retry while loading: {exc}')


# Download both at the same time (loading itself stays one at a time: it isn't thread-safe).
started = time.time()
with ThreadPoolExecutor(max_workers=2) as downloads:
    downloads.submit(_prefetch, snapshot_download, TEXT_ENCODER_ID, allow_patterns=['*.json', '*.safetensors'])
    # text_encoder=None: skips LTX's own (float32) text encoder.
    downloads.submit(_prefetch, LTXConditionPipeline.download, MODEL_ID, text_encoder=None)
print(f'downloaded in {time.time() - started:.0f} s')

started = time.time()
# Two GPUs: load the text encoder straight onto its GPU. One GPU: it waits in CPU memory.
_where = {} if SHARED_GPU else {'device_map': TEXT_DEVICE}
try:
    text_encoder = T5EncoderModel.from_pretrained(TEXT_ENCODER_ID, dtype=torch.bfloat16, **_where)
except Exception as exc:  # repo gone or renamed: use the original (bigger, slower) one
    print(f'{TEXT_ENCODER_ID} failed ({exc}); using the text encoder from {MODEL_ID}')
    text_encoder = T5EncoderModel.from_pretrained(MODEL_ID, subfolder='text_encoder', dtype=torch.bfloat16, **_where)
# Without a text encoder the pipeline runs only on VIDEO_DEVICE; prompts are encoded by encode() below.
# Loaded straight in MODEL_DTYPE: casting afterwards with .to(dtype) makes diffusers warn.
pipe = LTXConditionPipeline.from_pretrained(MODEL_ID, text_encoder=None, torch_dtype=MODEL_DTYPE)
pipe.vae.enable_tiling()
pipe.transformer.to(VIDEO_DEVICE)
pipe.vae.to(VIDEO_DEVICE)
print(f'loaded in {time.time() - started:.0f} s')
print(f'video model: {MODEL_DTYPE} on {VIDEO_DEVICE}; text encoder: bf16 on {TEXT_DEVICE}'
      + (' (takes turns with the video model)' if SHARED_GPU else ''))


def encode(text):
    """Same steps as the pipeline's own T5 encoding, but on TEXT_DEVICE and returned in MODEL_DTYPE."""
    tokens = pipe.tokenizer(text, padding='max_length', max_length=256, truncation=True,
                            add_special_tokens=True, return_tensors='pt')
    mask = tokens.attention_mask.bool()
    if SHARED_GPU:
        pipe.transformer.to('cpu'); pipe.vae.to('cpu'); torch.cuda.empty_cache(); text_encoder.to(TEXT_DEVICE)
    with torch.no_grad():
        embeds = text_encoder(tokens.input_ids.to(TEXT_DEVICE), attention_mask=mask.to(TEXT_DEVICE))[0]
    embeds = embeds.to(VIDEO_DEVICE, MODEL_DTYPE)
    if SHARED_GPU:
        text_encoder.to('cpu'); torch.cuda.empty_cache(); pipe.transformer.to(VIDEO_DEVICE); pipe.vae.to(VIDEO_DEVICE)
    return embeds, mask.to(VIDEO_DEVICE)


for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i) / 2**30
    total = torch.cuda.get_device_properties(i).total_memory / 2**30
    print(f'GPU {i} memory: {used:.1f} of {total:.1f} GB used')
print('Model ready')

In [ ]:
import gradio as gr

NEGATIVE = 'worst quality, inconsistent motion, blurry, jittery, distorted'
FPS = 24
MAX_SIDE = 512  # smaller = faster; must end up a multiple of 32
STEPS = 25
GUIDANCE = 5.0  # how closely the video follows the prompt (LTX docs recommend 5.0)


def _size(image):
    scale = MAX_SIDE / max(image.size)
    return [max(32, round(side * scale / 32) * 32) for side in image.size]


# Same name and parameters as the Lightricks/ltx-video-distilled Space, so the backend calls both the same way.
def image_to_video(prompt, input_image_filepath, mode='image-to-video', duration_ui=2,
                   progress=gr.Progress(track_tqdm=True)):
    started = time.time()
    image = Image.open(input_image_filepath).convert('RGB')
    width, height = _size(image)
    num_frames = int(float(duration_ui) * FPS) // 8 * 8 + 1  # LTX wants 8k+1 frames
    seed = random.randint(0, 2**31 - 1)
    embeds, mask = encode(prompt or 'The drawing comes to life with gentle playful motion, static camera')
    neg_embeds, neg_mask = encode(NEGATIVE)
    frames = pipe(
        prompt_embeds=embeds,
        prompt_attention_mask=mask,
        negative_prompt_embeds=neg_embeds,
        negative_prompt_attention_mask=neg_mask,
        image=image.resize((width, height)),
        width=width,
        height=height,
        num_frames=num_frames,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        decode_timestep=0.05,
        decode_noise_scale=0.025,
        generator=torch.Generator().manual_seed(seed),
    ).frames[0]
    if float(np.asarray(frames[len(frames) // 2]).std()) < 1.0:
        print('WARNING: the video looks blank.')
    out = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False).name
    export_to_video(frames, out, fps=FPS)
    print(f'video {width}x{height}, {num_frames} frames in {time.time() - started:.0f} s')
    return out, seed

In [ ]:
demo = gr.Interface(
    fn=image_to_video,
    inputs=[
        gr.Textbox(label='prompt'),
        gr.Image(type='filepath', label='input_image_filepath'),
        gr.Textbox(value='image-to-video', label='mode'),
        gr.Number(value=2, label='duration_ui'),
    ],
    outputs=[gr.Video(label='Generated Video'), gr.Number(label='Seed')],
    api_name='image_to_video',
    title='Doodle Alive GPU server (LTX-Video)',
)
demo.queue(max_size=20)
# show_error: the real error reaches the Doodle Alive backend and its flow chart
_, _, share_url = demo.launch(share=True, prevent_thread_lock=True, show_error=True)
print('=' * 70)
print('PASTE THIS LINK in the Doodle Alive admin panel (page URL + #admin):')
print(share_url)
print('=' * 70)
demo.block_thread()  # keeps the server running (needed for Kaggle 'Save & Run All')